In [1]:
import torch.optim as optim
from tqdm.auto import tqdm
import logging
import warnings
from collections import defaultdict
import pandas as pd
import gc
from transformers import (
    AutoImageProcessor,
    DinatForImageClassification,
    TrainingArguments,
    get_scheduler,
    AutoFeatureExtractor, SwinForImageClassification
)
from functions_D import *
from tqdm import tqdm
# from dashboard_functions import MultiSpeakerDashboard
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
# Suppress warnings and logging
logging.getLogger().addHandler(logging.NullHandler())
logging.getLogger("natten.functional").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

# Emotion mapping
EMOTIONS = {
    0: 'neutral',
    1: 'happy',
    2: 'sad',
    3: 'angry',
}
Map2Num = {
    'neutral': 0,
    'happy': 1,
    'sad': 2,
    'angry': 3,
}

# Configuration
num_labels = 4
base_column = "label"
dataset_train = "cairocode/IEMO_Mel_6"
dataset_val = "cairocode/MSPI_Mel6"
ds_tr = os.path.split(dataset_train)[1]
ds_vl = os.path.split(dataset_val)[1]

model_path = "shi-labs/dinat-mini-in1k-224"
checkpoint_path = '/media/carol/Data/Documents/Emo_rec/NewMel/Pretraining/Domination'
if checkpoint_path!= None:
    processor_path = os.path.join(checkpoint_path, 'processor')
    model_path = os.path.join(checkpoint_path, 'model')


pretrain_model = "microsoft/swin-base-patch4-window7-224" #model_path
BATCH_SIZE = 24

speaker_disentanglement = True
pretrain = True
column = "label"

# Loss function weights
alpha = 1.6
beta = 0.0008
gamma = 1

# Create output directory
base_dir = f"/media/carol/Data/Documents/Emo_rec/LOSO/{ds_tr}"
if pretrain and speaker_disentanglement:
    base_dir = os.path.join(base_dir, "PRSD")
elif pretrain:
    base_dir = os.path.join(base_dir, "PR")
elif speaker_disentanglement:
    base_dir = os.path.join(base_dir, "SD")
else:
    base_dir = os.path.join(base_dir, "OG")

base_dir = create_unique_output_dir(base_dir)
print(f"Output directory: {base_dir}")
os.makedirs(base_dir, exist_ok=True)

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Filter function
def filter_examples(example):
    return example["label"] != 4 and example["label"] != 5

# Load datasets
train_d0 = load_dataset(dataset_train, split='train')
val_dataset0 = load_dataset(dataset_val, split='train')

train_d0 = train_d0.filter(filter_examples)
val_dataset0 = val_dataset0.filter(filter_examples)
val_dataset0.set_transform(val_transforms)
# Set up cross-corpus dataloader
Xcorp_dataloader = DataLoader(
    val_dataset0,
    batch_size=BATCH_SIZE,
    collate_fn=lambda examples: collate_fn(examples),
)

# Get unique speakers for Leave-One-Speaker-Out (LOSO) validation
spkrs = [sample['speakerID'] for sample in train_d0]
unique_speakers = list(set(spkrs))
print(f"Unique speakers: {unique_speakers}")

# Initialize dictionaries for results
all_y_true = defaultdict(list)
all_y_pred = defaultdict(list)
total_results = pd.DataFrame()

xcorp_results = {
    'y_true': [],
    'y_pred': {f'run_{i}': [] for i in range(len(unique_speakers))}
}
# dashboard = MultiSpeakerDashboard(base_dir=base_dir, port=8000, auto_open=True)

# Start Leave-One-Speaker-Out (LOSO) training
for i in range(0, len(unique_speakers), 1):
    speakers = [unique_speakers[i]]
    num = speakers[0]
    print(f"\n {'#'*120}")
    print(f"                                          STARTING SPEAKER {num}                                                     ")
    print(f"\n {'#'*120}")

    # Create model output directory
    new_model_path = os.path.join(base_dir, str(num))
    os.makedirs(new_model_path, exist_ok=True)
    # dashboard.start_speaker_run(speaker_id=num, speaker_name=f"Speaker {num}")
    # print(f"Dashboard speaker info: {dashboard.current_speaker}, ID: {dashboard.current_speaker_id}")
    # Create the test split (left-out speaker)
    test_dataset = train_d0.filter(lambda x: x['speakerID'] in speakers).filter(filter_examples)

    # Create the training set (all other speakers)
    train_set = train_d0.filter(lambda x: x['speakerID'] not in speakers).filter(filter_examples)
    print("size ebefore balancing", len(train_set))
    train_set = balance_dataset(train_set, label_column="label", seed=42)
    print("AFETR", len(train_set))

    # Set transforms
    train_dataset = train_set
    val_dataset = test_dataset
    sd_sampler = CustomSampler(train_dataset)

    # Calculate class weights for balanced training
    # class_weights = calculate_class_weights(train_dataset)
    class_weights = [1.13, 2.8, 1.0 ,1.0]
    # class_weights[0] = 1.2
    # class_weights[1] = 1.2

    # class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    # print(f"Class weights: {class_weights}")

    train_set.set_transform(train_transforms)
    custom_dataset = CustomDataset(train_set)
    val_dataset.set_transform(val_transforms)
    test_dataset.set_transform(val_transforms)

    # Set up data loaders
    if speaker_disentanglement:
        print("HELLOOOOO")
        train_sampler = sd_sampler
        train_loader = DataLoader(
            custom_dataset,
            sampler=train_sampler,
            batch_size=BATCH_SIZE,
            collate_fn=lambda examples: collate_fn(examples),
        )
    else:
        print("BYEEEEEEEE")

        train_loader = DataLoader(
            custom_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=lambda examples: collate_fn(examples),
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda examples: collate_fn(examples),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda examples: collate_fn(examples),
    )


    # Load DiNAT model
    base_model = SwinForImageClassification.from_pretrained(
        pretrain_model,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
        problem_type="single_label_classification",
    ).to(device)

    # Create model with feature extraction capabilities
    model = DiNATWithFeatures(
        pretrained_model=base_model,
        num_classes=num_labels,
        feature_dim=1024
    ).to(device)

    training_args = TrainingArguments(
        output_dir="./logs",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=50,
        weight_decay=0.05,
        load_best_model_at_end=True
    )

    # Set up loss function
    cecc_loss = BalancedCrossEntropyWithContrastiveLoss(
        num_classes=num_labels,
        feature_dim=1024,
        class_weight_multipliers = class_weights,
    )

    # # Optimizer
    optimizer = optim.AdamW([
        {'params': model.parameters(), 'weight_decay': training_args.weight_decay},
        {'params': cecc_loss.parameters(), 'lr': 0.0001, 'weight_decay': 0.01}
    ], 
        lr=training_args.learning_rate
    )


    # Learning rate scheduler
    num_training_steps = len(train_loader) * training_args.num_train_epochs
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    # from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    # lr_scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

    # Cosine annealing with restarts


    # Training loop parameters
    num_epochs = training_args.num_train_epochs
    patience = 10
    best_val_uar = 0
    patience_counter = 0
    train_losses, val_losses, epochs_list = [], [], []
    best_model_path = os.path.join(new_model_path, "best_model.pt")
    
    # Begin training
    for epoch in range(int(num_epochs)):
        model.train()
        train_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        batch_idx = 0 
        for batch in progress_bar:
            batch_idx+=1
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs["logits"]
            features = outputs["features"]

            loss, weight_info = cecc_loss(logits, features, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            progress_bar.set_postfix({"Loss": loss.item()})

            # if batch_idx % 5 == 0:
            #     dashboard.update_batch(
            #         epoch=epoch+1,
            #         batch=batch_idx+1,
            #         train_loss=loss.item(),
            #         total_batches=len(train_loader)
            #     )
            

        avg_train_loss = train_loss / len(train_loader)
        lr_scheduler.step()

        # Validation
        model.eval()
        val_loss = 0
        all_predictions, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch["pixel_values"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(pixel_values=pixel_values)
                logits = outputs["logits"]
                features = outputs["features"]

                loss, weight_info = cecc_loss(logits, features, labels)
                val_loss += loss.item()

                predictions = torch.argmax(logits, dim=-1)
                all_predictions.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        accuracy = accuracy_score(all_labels, all_predictions)
        uar = recall_score(all_labels, all_predictions, average="macro")
        f1 = f1_score(all_labels, all_predictions, average="macro")
        per_class_recall = recall_score(all_labels, all_predictions, average=None)
        uar_std = np.std(per_class_recall)

        gamma_comp = 1.5
        comparison_metric = uar / (1 + gamma_comp * uar_std)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        epochs_list.append(epoch + 1)

        print(
            f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_train_loss:.4f}, "
            f"Validation Loss: {avg_val_loss:.4f}, "
            f"Accuracy: {accuracy:.4f}, UAR: {uar:.4f}, F1: {f1:.4f}, UAR STD: {uar_std:.4f}, "
            f"Comparison metric: {comparison_metric:.4f}\n"
            f"CE weight: {weight_info['weight_ce']:.6f} (log var: {weight_info['log_var_ce']:.4f}), "
            f"Contrastive weight: {weight_info['weight_contrastive']:.6f} (log var: {weight_info['log_var_contrastive']:.4f}), "
            f"Balance weight: {weight_info['weight_balance']:.6f} (log var: {weight_info['log_var_balance']:.4f})"
            f"Base Gamma: {weight_info['base_gamma']:.4f}  Class Weights: {[f'{w:.4f}' for w in weight_info['class_weights']]}  Class Gammas: {[f'{g:.4f}' for g in weight_info['class_gammas']]}"
        )
        # dashboard.update(
        #     epoch=epoch+1,
        #     train_loss=avg_train_loss,
        #     val_loss=avg_val_loss,
        #     accuracy=accuracy,
        #     uar=uar,
        #     f1=f1
        # )

        # Early Stopping based on comparison metric
        if uar > best_val_uar:
            best_val_uar = uar
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            best_epoch = epoch
            print("Validation uar improved. Best model saved.")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    # Load Best Model
    print("Loading best model for final evaluation.")
    model.load_state_dict(torch.load(best_model_path))
    model.to(device)

    ##############################################################################
    # Test Evaluation
    ##############################################################################
    print("\nStarting Test Evaluation...")
    model.eval()
    test_loss = 0
    all_test_predictions, all_test_labels = [], []

    with torch.no_grad():
        test_progress_bar = tqdm(test_loader, desc="Testing", leave=False)
        for batch in test_progress_bar:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs["logits"]

            loss = F.cross_entropy(logits, labels)
            test_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            all_test_predictions.extend(predictions.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())

    avg_test_loss = test_loss / len(test_loader)
    test_accuracy = accuracy_score(all_test_labels, all_test_predictions)
    test_uar = recall_score(all_test_labels, all_test_predictions, average="macro")
    test_f1 = f1_score(all_test_labels, all_test_predictions, average="macro")

    metrics_str = (
        f"Test Loss: {avg_test_loss:.4f}, "
        f"Accuracy: {test_accuracy:.4f}, "
        f"UAR: {test_uar:.4f}, "
        f"F1: {test_f1:.4f}"
    )
    print(metrics_str)

    # Save confusion matrix
    plot_and_save_confusion_matrix(
        all_test_labels, 
        all_test_predictions, 
        list(EMOTIONS.values()), 
        new_model_path, 
        filename=f"{ds_tr}_{test_accuracy:.4f}_Acc_{test_uar:.4f}_UAR.png"
    )
    
    # Save metrics to file
    output_file = os.path.join(new_model_path, "metrics.txt")
    with open(output_file, "w") as f:
        f.write(f"F1 Score: {test_f1:.4f}\n")
        f.write(f"Accuracy: {test_accuracy:.4f}\n")
        f.write(f"UAR: {test_uar:.4f}\n")
        f.write(f"Class Mapping: {EMOTIONS}\n")
        f.write(f"Best Epoch: {best_epoch}\n")
        f.write(f"Train Dataset: {ds_tr}\n")
        f.write(f"Alpha: {alpha}\n")
        f.write(f"Beta: {beta}\n")
        f.write(f"Gamma: {gamma}\n")
        f.write(f"Class Weights: {class_weights}\n")
    print(f"Metrics saved to {output_file}")
    
    #############################################################################
    # CROSS CORPUS Evaluation
    ##############################################################################
    print("\nStarting Cross Corpus Evaluation...")
    model.eval()
    test_loss = 0
    X_preds, X_true = [], []

    with torch.no_grad():
        test_progress_bar = tqdm(Xcorp_dataloader, desc="Cross-Corpus Testing", leave=False)
        for batch in test_progress_bar:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs["logits"]

            loss = F.cross_entropy(logits, labels)
            test_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            X_preds.extend(predictions.cpu().numpy())
            X_true.extend(labels.cpu().numpy())

    Xavg_test_loss = test_loss / len(Xcorp_dataloader)
    Xtest_accuracy = accuracy_score(X_true, X_preds)
    Xtest_uar = recall_score(X_true, X_preds, average="macro")
    Xtest_f1 = f1_score(X_true, X_preds, average="macro")

    if i == 0:
        xcorp_results['y_true'].extend(X_true)

    # Store y_pred for each run
    xcorp_results['y_pred'][f'run_{i}'].extend(X_preds)

    metrics_str = (
        f"Cross-Corpus Test Loss: {Xavg_test_loss:.4f}, "
        f"Accuracy: {Xtest_accuracy:.4f}, "
        f"UAR: {Xtest_uar:.4f}, "
        f"F1: {Xtest_f1:.4f}"
    )
    print(metrics_str)
    
    # Save cross-corpus confusion matrix
    plot_and_save_confusion_matrix(
        X_true, 
        X_preds, 
        list(EMOTIONS.values()), 
        new_model_path, 
        filename=f"{ds_vl}_{Xtest_accuracy:.4f}_Acc_{Xtest_uar:.4f}_UAR.png"
    )
    
    # Update results for averaging
    new_row = {
        f'{ds_tr}_ACC': test_accuracy*100, 
        f'{ds_tr}_UAR': test_uar*100, 
        f'{ds_vl}_ACC': Xtest_accuracy*100,
        f'{ds_vl}_UAR': Xtest_uar*100
    }
    print(f"\n {'-'*80}")
    print("\nResults:", new_row, "\n")
    
    total_results = pd.concat([total_results, pd.DataFrame([new_row])], ignore_index=True)
    
    # Save cross-corpus metrics
    output_file = os.path.join(new_model_path, f"{ds_vl}_metrics.txt")
    with open(output_file, "w") as f:
        f.write(f"F1 Score: {Xtest_f1:.4f}\n")
        f.write(f"Accuracy: {Xtest_accuracy:.4f}\n")
        f.write(f"UAR: {Xtest_uar:.4f}\n")
        f.write(f"Class Mapping: {EMOTIONS}\n")
        f.write(f"Best Epoch: {best_epoch}\n")

    print(f"Cross-corpus metrics saved to {output_file}")

    # Keep track of results across all speakers
    all_y_true[ds_tr].extend(all_test_labels)
    all_y_pred[ds_tr].extend(all_test_predictions)
    all_y_true[ds_vl].extend(X_true)
    all_y_pred[ds_vl].extend(X_preds)
    # dashboard._archive_current_run()

    # Clean up to prevent memory issues
    torch.cuda.empty_cache()
    del model

# Calculate final results
print("\n====================== FINAL RESULTS ======================\n")

# If you want to get the final prediction based on majority voting across all runs
if len(xcorp_results['y_true']) > 0:
    y_true = np.array(xcorp_results['y_true'])
    
    # Create majority vote predictions
    run_count = len(unique_speakers)
    final_prediction = np.array([
        np.bincount([xcorp_results['y_pred'][f'run_{i}'][j] for i in range(run_count) if j < len(xcorp_results['y_pred'][f'run_{i}'])], 
                    minlength=num_labels).argmax() 
        for j in range(len(y_true))
    ])
    
    final_accuracy = accuracy_score(y_true, final_prediction)
    final_uar = recall_score(y_true, final_prediction, average="macro")
    print(f"Final cross-corpus accuracy after majority voting: {final_accuracy:.4f}")
    print(f"Final cross-corpus UAR after majority voting: {final_uar:.4f}")

# Print overall results
print("\nDetailed Results:")
print(total_results)

# Calculate average metrics
avg_results = total_results.mean(numeric_only=True)
print("\nAVERAGE RESULTS\n", avg_results)

# Calculate final metrics for both datasets
final_metrics = {}
for dataset in [ds_tr, ds_vl]:
    if len(all_y_true[dataset]) > 0:
        y_true = np.array(all_y_true[dataset])
        y_pred = np.array(all_y_pred[dataset])
        
        acc = accuracy_score(y_true, y_pred) * 100
        uar = recall_score(y_true, y_pred, average="macro") * 100
        f1 = f1_score(y_true, y_pred, average="macro") * 100
        final_metrics[f'{dataset}_ACC'] = acc
        final_metrics[f'{dataset}_UAR'] = uar
        final_metrics[f'{dataset}_F1'] = f1


# dashboard.export_results(base_dir)


/media/carol/Data/Documents/Emo_rec/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output directory: /media/carol/Data/Documents/Emo_rec/LOSO/IEMO_Mel_6/PRSD/20250420_12
Using device: cuda
Unique speakers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

 ########################################################################################################################
                                          STARTING SPEAKER 1                                                     

 ########################################################################################################################
size ebefore balancing 4025
Regular Dataset Length: 4025 -- Balanced Dataset Length: 4024
AFETR 4024
HELLOOOOO


/media/carol/Data/Documents/Emo_rec/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-base-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([4, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 - Training Loss: 6.7426, Validation Loss: 5.4072, Accuracy: 0.4301, UAR: 0.5228, F1: 0.4243, UAR STD: 0.2808, Comparison metric: 0.3678
CE weight: 0.206634 (log var: -1.9862), Contrastive weight: 4.257331 (log var: 3.0128), Balance weight: 1.553042 (log var: -1.5854)Base Gamma: 5.1140  Class Weights: ['1.1301', '2.7829', '1.0149', '1.0128']  Class Gammas: ['1.4140', '1.8128', '1.0142', '1.0139']
Validation uar improved. Best model saved.


Epoch 2/50 - Training Loss: 4.7535, Validation Loss: 4.3450, Accuracy: 0.4796, UAR: 0.5449, F1: 0.4432, UAR STD: 0.3406, Comparison metric: 0.3607
CE weight: 0.241151 (log var: -1.9744), Contrastive weight: 3.456135 (log var: 3.0229), Balance weight: 0.534250 (log var: -1.5710)Base Gamma: 5.1264  Class Weights: ['1.1361', '2.7655', '1.0316', '1.0284']  Class Gammas: ['1.4265', '1.8241', '1.0265', '1.0252']
Validation uar improved. Best model saved.


...

Test Loss: 0.9959, Accuracy: 0.6611, UAR: 0.5932, F1: 0.5702
Confusion matrix saved to: /media/carol/Data/Documents/Emo_rec/LOSO/IEMO_Mel_6/PRSD/20250420_12/8/IEMO_Mel_6_0.6611_Acc_0.5932_UAR.png
Metrics saved to /media/carol/Data/Documents/Emo_rec/LOSO/IEMO_Mel_6/PRSD/20250420_12/8/metrics.txt

Starting Cross Corpus Evaluation...


Cross-Corpus Test Loss: 1.2319, Accuracy: 0.3725, UAR: 0.3903, F1: 0.3210
Confusion matrix saved to: /media/carol/Data/Documents/Emo_rec/LOSO/IEMO_Mel_6/PRSD/20250420_12/8/MSPI_Mel6_0.3725_Acc_0.3903_UAR.png

 --------------------------------------------------------------------------------

Results: {'IEMO_Mel_6_ACC': 66.10978520286396, 'IEMO_Mel_6_UAR': 59.316095080318796, 'MSPI_Mel6_ACC': 37.25314183123878, 'MSPI_Mel6_UAR': 39.03356262959756} 

Cross-corpus metrics saved to /media/carol/Data/Documents/Emo_rec/LOSO/IEMO_Mel_6/PRSD/20250420_12/8/MSPI_Mel6_metrics.txt

 ########################################################################################################################
                                          STARTING SPEAKER 9                                                     

 ########################################################################################################################
size ebefore balancing 3982
Regular Dataset Length: 3982 -- Balanc

/media/carol/Data/Documents/Emo_rec/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-base-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([4, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 - Training Loss: 6.5681, Validation Loss: 5.4206, Accuracy: 0.2618, UAR: 0.3938, F1: 0.2621, UAR STD: 0.3227, Comparison metric: 0.2653
CE weight: -0.289648 (log var: -1.9864), Contrastive weight: 4.232367 (log var: 3.0126), Balance weight: -0.792829 (log var: -1.5857)Base Gamma: 5.1142  Class Weights: ['1.1313', '2.7835', '1.0151', '1.0133']  Class Gammas: ['1.4144', '1.8121', '1.0140', '1.0142']
Validation uar improved. Best model saved.


OutOfMemoryError: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 

In [ ]:

print(f"                                          STARTING CROSS CORPUS FULL TRAINING                                                    ")
print(f"\n {'#'*120}")

# Create model output directory
new_model_path = os.path.join(base_dir, ds_vl)
os.makedirs(new_model_path, exist_ok=True)

test_dataset = val_dataset0

# Create the training set (all other speakers)
train_set = train_d0
train_dataset = train_set
val_dataset = test_dataset
sd_sampler = CustomSampler(train_dataset)

class_weights = [0.95,2,1.1,1]

# class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

train_set.set_transform(train_transforms)
custom_dataset = CustomDataset(train_set)
val_dataset.set_transform(val_transforms)
test_dataset.set_transform(val_transforms)

# Set up data loaders
if speaker_disentanglement:
    print("HELLOOOOO")
    train_sampler = sd_sampler
    train_loader = DataLoader(
        custom_dataset,
        sampler=train_sampler,
        batch_size=BATCH_SIZE,
        collate_fn=lambda examples: collate_fn(examples),
    )
else:
    print("BYEEEEEEEE")

    train_loader = DataLoader(
        custom_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda examples: collate_fn(examples),
    )

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda examples: collate_fn(examples),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda examples: collate_fn(examples),
)


# Load DiNAT model
base_model = DinatForImageClassification.from_pretrained(
    pretrain_model,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification",
).to(device)

# Create model with feature extraction capabilities
model = DiNATWithFeatures(
    pretrained_model=base_model,
    num_classes=num_labels,
    feature_dim=1024
).to(device)

training_args = TrainingArguments(
    output_dir="./logs",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=50,
    weight_decay=0.05,
    load_best_model_at_end=True
)

# Set up loss function
cecc_loss = BalancedCrossEntropyWithContrastiveLoss(
        num_classes=num_labels,
        feature_dim=512,
        class_weight_multipliers = class_weights
    )

# # Optimizer
optimizer = optim.AdamW([
    {'params': model.parameters(), 'weight_decay': training_args.weight_decay},
    {'params': cecc_loss.parameters(), 'lr': 0.001, 'weight_decay': 0.01}
], 
    lr=training_args.learning_rate
)


# Learning rate scheduler
num_training_steps = len(train_loader) * training_args.num_train_epochs
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

num_epochs = training_args.num_train_epochs
patience = 10
best_val_uar = 0
patience_counter = 0
train_losses, val_losses, epochs_list = [], [], []
best_model_path = os.path.join(new_model_path, "best_model.pt")

# Begin training
for epoch in range(int(num_epochs)):
    model.train()
    train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
    batch_idx = 0 
    for batch in progress_bar:
        batch_idx+=1
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values)
        logits = outputs["logits"]
        features = outputs["features"]

        loss, weight_info = cecc_loss(logits, features, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        progress_bar.set_postfix({"Loss": loss.item()})


    avg_train_loss = train_loss / len(train_loader)
    lr_scheduler.step()

    # Validation
    model.eval()
    val_loss = 0
    all_predictions, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs["logits"]
            features = outputs["features"]

            loss, weight_info = cecc_loss(logits, features, labels)
            val_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_predictions)
    uar = recall_score(all_labels, all_predictions, average="macro")
    f1 = f1_score(all_labels, all_predictions, average="macro")
    per_class_recall = recall_score(all_labels, all_predictions, average=None)
    uar_std = np.std(per_class_recall)

    gamma_comp = 1.5
    comparison_metric = uar / (1 + gamma_comp * uar_std)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    epochs_list.append(epoch + 1)

    print(
        f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_train_loss:.4f}, "
        f"Validation Loss: {avg_val_loss:.4f}, "
        f"Accuracy: {accuracy:.4f}, UAR: {uar:.4f}, F1: {f1:.4f}, UAR STD: {uar_std:.4f}, "
        f"Comparison metric: {comparison_metric:.4f}\n"
        f"CE weight: {weight_info['weight_ce']:.6f} (log var: {weight_info['log_var_ce']:.4f}), "
        f"Contrastive weight: {weight_info['weight_contrastive']:.6f} (log var: {weight_info['log_var_contrastive']:.4f}), "
        f"Balance weight: {weight_info['weight_balance']:.6f} (log var: {weight_info['log_var_balance']:.4f})"
        f"Base Gamma: {weight_info['base_gamma']:.4f}  Class Weights: {[f'{w:.4f}' for w in weight_info['class_weights']]}  Class Gammas: {[f'{g:.4f}' for g in weight_info['class_gammas']]}"
    )
    if uar > best_val_uar:
        best_val_uar = uar
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        best_epoch = epoch
        print("Validation uar improved. Best model saved.")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Load Best Model
print("Loading best model for final evaluation.")
model.load_state_dict(torch.load(best_model_path))
model.to(device)

##############################################################################
# Test Evaluation
##############################################################################
print("\nStarting Test Evaluation...")
model.eval()
test_loss = 0
all_test_predictions, all_test_labels = [], []

with torch.no_grad():
    test_progress_bar = tqdm(test_loader, desc="Testing", leave=False)
    for batch in test_progress_bar:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values)
        logits = outputs["logits"]

        loss = F.cross_entropy(logits, labels)
        test_loss += loss.item()

        predictions = torch.argmax(logits, dim=-1)
        all_test_predictions.extend(predictions.cpu().numpy())
        all_test_labels.extend(labels.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
test_accuracy = accuracy_score(all_test_labels, all_test_predictions)
test_uar = recall_score(all_test_labels, all_test_predictions, average="macro")
test_f1 = f1_score(all_test_labels, all_test_predictions, average="macro")

metrics_str = (
    f"Test Loss: {avg_test_loss:.4f}, "
    f"Accuracy: {test_accuracy:.4f}, "
    f"UAR: {test_uar:.4f}, "
    f"F1: {test_f1:.4f}"
)
print(metrics_str)

# Save confusion matrix
plot_and_save_confusion_matrix(
    all_test_labels, 
    all_test_predictions, 
    list(EMOTIONS.values()), 
    new_model_path, 
    filename=f"{ds_vl}_{test_accuracy:.4f}_Acc_{test_uar:.4f}_UAR.png"
)

# Save metrics to file
output_file = os.path.join(new_model_path, "metrics.txt")
with open(output_file, "w") as f:
    f.write(f"F1 Score: {test_f1:.4f}\n")
    f.write(f"Accuracy: {test_accuracy:.4f}\n")
    f.write(f"UAR: {test_uar:.4f}\n")
    f.write(f"Class Mapping: {EMOTIONS}\n")
    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"Train Dataset: {ds_tr}\n")
    f.write(f"Alpha: {alpha}\n")
    f.write(f"Beta: {beta}\n")
    f.write(f"Gamma: {gamma}\n")
    f.write(f"Class Weights: {class_weights}\n")
print(f"Metrics saved to {output_file}")


In [ ]:
print("\nFinal Aggregate Metrics:")
print(final_metrics)

# Save overall model info
model_info = {
    "Pretrain_file": checkpoint_path if checkpoint_path else "None",
    "Dataset Used": dataset_train,
    "Model Type": "DiNAT-only",
    "Speaker Disentanglement": speaker_disentanglement,
    "Column Trained on": column,
    "Average Results": avg_results.to_dict(), 
    "Final Metrics": final_metrics,
    "Alpha": alpha,
    "Beta": beta,
    "Gamma": gamma,
    'total_results': total_results
}

save_model_info_to_txt(model_info, base_dir)
print(f"Summary results saved to {base_dir}/results.txt")
print("\nTraining complete!")
# dashboard.finish()

In [ ]:
class_weights

In [ ]:
train_dataset

In [ ]:
train_set

In [ ]:
# sudo lsof -i :8000    # Find the process ID (PID)
# sudo kill -9 [PID]    # Kill the process

In [ ]:
# Add this before creating the DataLoader
print(f"Dataset type: {type(train_dataset)}")
print(f"Dataset length: {len(train_dataset) if hasattr(train_dataset, '__len__') else 'unknown'}")
print(f"Custom dataset type: {type(custom_dataset)}")
print(f"Custom dataset length: {len(custom_dataset) if hasattr(custom_dataset, '__len__') else 'unknown'}")

# Add this to your CustomDataset.__getitem__ method for debugging
def __getitem__(self, idx):
    if self.dataset is None:
        print(f"Dataset is None at idx {idx}")
        # Return a default value or raise a more informative error
        raise ValueError(f"Dataset is None in CustomDataset.__getitem__ at idx {idx}")
    
    try:
        item = self.dataset[idx]
        return item
    except Exception as e:
        print(f"Error accessing dataset at idx {idx}: {e}")
        raise

In [ ]:
# Print more details about the datasets
print(f"train_set length: {len(train_set)}")
print(f"train_dataset before transform: {type(train_dataset)} with length {len(train_dataset)}")
print(f"train_dataset after transform: {type(train_dataset)} with length {len(train_dataset)}")

# Check if train_dataset is None
if train_dataset is None:
    print("ERROR: train_dataset is None before creating CustomDataset!")

In [ ]:
train_dataset

In [ ]:
test_dataset